# 🎨 Image Restoration Dual-Model Training - Colab

This notebook supports **Deterministic (SwinIR)** and **Generative (SD-LoRA)** restoration with automatic **Benchmarking**.

### ✅ Requirements:
Ensure you have uploaded **ALL THREE** `restoration_code.zip`, `processed_data.zip`, and `processed_sd.zip` to the root of your Google Drive.

### 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Create directories for weights and validation logs
!mkdir -p /content/drive/MyDrive/weights/swinir_checkpoints
!mkdir -p /content/drive/MyDrive/weights/sd_lora_val_images

### 2. Smart Extract: Code & Data

In [ ]:
# 1. Clone Source Code from GitHub directly to Google Drive
print("📁 Pulling Code from Github to Drive...")
%cd /content/drive/MyDrive/
!rm -rf Image-Restoration
!git clone -b cauhoiontap https://github.com/alitonia/Image-Restoration.git
!cp -r Image-Restoration/training /content/training
%cd Image-Restoration


In [ ]:
# 2. Extract Large Datasets
if os.path.exists('/content/drive/MyDrive/processed_data.zip'):
    print("📊 Extracting Standard Processed Dataset...")
    !unzip -q /content/drive/MyDrive/processed_data.zip -d /content/



In [ ]:
if os.path.exists('/content/drive/MyDrive/processed_sd.zip'):
    print("🎨 Extracting SD-Optimized Dataset...")
    !unzip -q /content/drive/MyDrive/processed_sd.zip -d /content/
else:
    print("⚠️ 'processed_sd.zip' not found. Run prepare_lite_sd.sh if needed.")


In [ ]:
%cd /content/training

# Install dependencies (fast cloud install)
!pip install -q -r requirements.txt
!pip install -q xformers peft accelerate diffusers transformers

In [ ]:
!pip install mediapipe==0.10.14

### 3. Data Preparation (ONLY if processed_data.zip was missing)
This generates clean/damaged pairs and separates them into `train` and `test` sets automatically.

In [ ]:
# !python scripts/prepare_data.py --input ./datasets/raw_lite \
#                               --output ./datasets/processed \
#                               --seed 42 \
#                               --split_ratio 0.9 \
#                               --multiplier 1

### 4. Track 1: SwinIR-Light Training (Sanity Test)
Run a quick 1-epoch test to ensure the dataset is loaded correctly and CUDA is working before the full run.

In [ ]:
!python scripts/train_swinir.py --clean_dir ./datasets/processed/train/clean \
                                --damaged_dir ./datasets/processed/train/damaged \
                                --epochs 1 \
                                --batch_size 2 \
                                --weight_path /content/drive/MyDrive/weights/swinir_restoration_test.pth

### 4.5 Track 1: SwinIR-Light Training (Full Run)
The model will periodically evaluate on the test set and save the best PSNR weights.

In [ ]:
!python scripts/train_swinir.py --clean_dir ./datasets/processed/train/clean \
                                --damaged_dir ./datasets/processed/train/damaged \
                                --val_clean_dir ./datasets/processed/test/clean \
                                --val_damaged_dir ./datasets/processed/test/damaged \
                                --epochs 100 \
                                --batch_size 16 \
                                --patch_size 128 \
                                --weight_path /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                --resume

### 5. Track 2: SD-LoRA Transfer Training (Sanity Test)
Run a quick 1-epoch test with a tiny SD model to ensure the pipeline works.

In [ ]:
!python scripts/train_sd_lora.py --image_dir ./datasets/processed/train/clean \
                                 --output_dir /content/drive/MyDrive/weights/sd_lora_antique_test \
                                 --epochs 1 \
                                 --batch_size 1 \
                                 --model_id hf-internal-testing/tiny-stable-diffusion-torch

### 5.5 Track 2: SD-LoRA Transfer Training (Full Run)
Check `/content/drive/MyDrive/weights/sd_lora_val_images` to see restoration progress.

In [ ]:
!python scripts/train_sd_lora.py --image_dir ./datasets/processed/train/clean \
                                 --output_dir /content/drive/MyDrive/weights/sd_lora_antique_colab \
                                 --val_dir /content/drive/MyDrive/weights/sd_lora_val_images \
                                 --epochs 20 \
                                 --batch_size 4 \
                                 --resume

### 6. Track 3: Benchmarking
Evaluate both networks computationally against the ground truth, and generate an image grid comparing the outputs.

In [ ]:
### 6.1 Evaluate Deterministic SwinIR Model
print("📊 Running Quantitative Metrics (PSNR/SSIM/MSE/NRMSE) for SwinIR...")
!python scripts/evaluate.py --clean_dir ./datasets/processed/test/clean \
                            --damaged_dir ./datasets/processed/test/damaged \
                            --weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth


In [ ]:
### 6.2 Evaluate Generative SD-LoRA Model
print("📊 Running Quantitative Metrics for SD-LoRA...")
print("(Note: This generates the full dataset using ControlNet+LoRA so it takes a little longer!)")
!python scripts/evaluate.py --clean_dir ./datasets/processed/test/clean \
                            --damaged_dir ./datasets/processed/test/damaged \
                            --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest


In [ ]:
### 6.3 Generate Visual Side-by-Side Comparison Grid
print("🖼️ Generating visual comparison grid...")
!python scripts/test_restoration.py --test_clean_dir ./datasets/processed/test/clean \
                                    --test_damaged_dir ./datasets/processed/test/damaged \
                                    --swinir_weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                    --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest \
                                    --output_dir ./test_results \
                                    --num_test 10


In [ ]:
# Display results
from IPython.display import Image, display
import glob
grids = glob.glob('./test_results/*.png')
if grids:
    display(Image(filename=grids[0]))

In [ ]:
# Display all generated result grids
from IPython.display import Image, display
import glob
grids = sorted(glob.glob('./test_results/*.png'))
if grids:
    for grid in grids:
        display(Image(filename=grid))
else:
    print("No test results found.")


### 7. Iterative/Recursive Restoration Testing
Pass test images multiple times through the models to see the compound effect (e.g., n=10 passes).

In [ ]:
### 7.1 Run Recursive Test (N=10 passes)
print("🔄 Running recursive inference (this will take a while for SD-LoRA!)...")
!python scripts/test_recursive.py --test_damaged_dir ./datasets/processed/test/damaged \
                                  --swinir_weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                  --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest \
                                  --output_dir ./recursive_results \
                                  --num_test 2 \
                                  --passes 10


In [ ]:
# Display the Iterative GIF/Grid Results
from IPython.display import Image, display
import glob
print("SwinIR Effect:")
swinir_gifs = sorted(glob.glob('./recursive_results/*_swinir_effect.gif'))
for gif in swinir_gifs:
    display(Image(filename=gif))

print("SD-LoRA Effect:")
sd_gifs = sorted(glob.glob('./recursive_results/*_sd_effect.gif'))
for gif in sd_gifs:
    display(Image(filename=gif))


In [ ]:
### 7.2 Run Recursive Test SUPER (N=30 passes)
print("🔄 Running recursive inference (this will take a while for SD-LoRA!)...")
!python scripts/test_recursive.py --test_damaged_dir ./datasets/processed/test/damaged \
                                  --swinir_weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                  --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest \
                                  --output_dir ./recursive_results_super \
                                  --num_test 2 \
                                  --passes 30

In [ ]:
# Display the Iterative GIF/Grid Results
from IPython.display import Image, display
import glob
print("SwinIR Effect:")
swinir_gifs = sorted(glob.glob('./recursive_results_super/*_swinir_effect.gif'))
for gif in swinir_gifs:
    display(Image(filename=gif))

print("SD-LoRA Effect:")
sd_gifs = sorted(glob.glob('./recursive_results_super/*_sd_effect.gif'))
for gif in sd_gifs:
    display(Image(filename=gif))


In [ ]:
# Test model logic explicitly using GPU in Colab to verify pipeline runs optimally (Added by request)
!python scripts/test_model.py \
        --clean_dir ./datasets/processed/train/clean \
        --damaged_dir ./datasets/processed/train/damaged
